# SPHEREx Golden Path: how the API processes the `.fits` files

Minimal, reproducible version of what `spherex-api` does for the 5 challenge targets.
It reuses the backend's own pipeline (`scripts/generate_golden_path.py`), so the numbers here are the ones the API serves.

Data: SPHEREx Level 2 Spectral Images from IRSA (`spherex_dr1` + `spherex_qr3`). Needs network access to IRSA and
`pip install -r requirements-science.txt`. Run it from the repository root (or copy it to `spherex-science-notebooks`
together with the `app/` and `scripts/` folders).

Steps: (1) find every visit, (2) inspect one real cutout (MEF extensions, FLAGS bit names, spectral WCS),
(3) build per-visit, per-detector coadds, the difference image and the spectrum.

In [ ]:
import importlib.util, sys, tempfile
from pathlib import Path
import numpy as np
import astropy.units as u
from astropy.coordinates import SkyCoord
from astropy.io import fits
from astropy.time import Time
from astropy.wcs import WCS

ROOT = Path.cwd() if (Path.cwd() / "scripts").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
spec = importlib.util.spec_from_file_location("gp", ROOT / "scripts" / "generate_golden_path.py")
gp = importlib.util.module_from_spec(spec); sys.modules["gp"] = gp; spec.loader.exec_module(gp)
gp.quiet_astropy()
from app.core.targets import CATALOG
target = CATALOG["WISE_0855"]   # try BARNARDS_STAR, EPSILON_INDI_B, RW_AURIGAE, R_DRACONIS

## 1. Every visit covering the target

WISE 0855 moves ~8"/yr, so its position is propagated (proper motion) before querying.
Exposures are grouped into **visits** by gaps in `t_min` (MJD), never by fixed calendar semesters
(a fixed semester boundary split real visits in two).

In [ ]:
coord = gp.position_at(target, Time("2026-01-01"))
rows = gp.dedupe(gp.query_sia(coord, gp.WIDE_COLLECTIONS, budget_s=120), gp.WIDE_COLLECTIONS)
print(len(rows), "exposures;", sorted(set(rows["obs_collection"])))
print(rows["obs_id", "energy_bandpassname", "t_min", "obs_collection"][:5])
visits = gp.pick_epochs(gp.group_epochs(rows, 30.0), "all", 12)
for v in gp.summarize_visits(visits):
    print(v["epoch_id"], v["date_start"], "->", v["date_end"], v["n_images"], "exposures", v["n_per_detector"])

## 2. One real cutout

IRSA's cutout service (`access_url?center=RA,DEC&size=DEG`) returns the same MEF layout as the full 2040x2040 frame:
IMAGE (MJy/sr, zodiacal light **not** subtracted), FLAGS, VARIANCE, ZODI, EPSF, WCS-WAVE.
The wavelength of each pixel comes from the spectral WCS (`key="W"`, SIP turned off), as in the IRSA tutorials.

In [ ]:
row = visits[0][0]
url = f"{row['access_url']}?center={coord.ra.deg:.6f},{coord.dec.deg:.6f}&size={6 / 60:.4f}"
with fits.open(url) as h:
    h.info()
    flag_names = gp.decode_flag_names(h["FLAGS"].header)
    print("FLAGS bits:", flag_names)
    spectral = WCS(h["IMAGE"].header, fobj=h, key="W"); spectral.sip = None
    ny, nx = h["IMAGE"].data.shape
    wl, bw = spectral.pixel_to_world(nx / 2, ny / 2)
    print(f"{row['energy_bandpassname']}: centre wavelength {wl.to(u.um):.4f}, bandwidth {bw.to(u.um):.4f}")

## 3. Coadds, difference image, spectrum

`build_payload` runs the whole pipeline: cutouts in parallel, `IMAGE - ZODI`, flag masking, reprojection onto a
common north-up grid, a **median coadd per visit and per detector** (each detector sees different wavelengths, so
detectors are never subtracted from each other), last-minus-first difference, OpenCV anomaly detection on the
empirical-sigma SNR map, and a 6 x 17 = 102-channel spectrum.

`max_per_detector=2` keeps this quick (~1-2 min); the cached Golden Path uses every exposure.

In [ ]:
from IPython.display import Image, display
out = Path(tempfile.mkdtemp())

class TempStorage(gp.StorageUploader):
    def upload_png(self, filename, data):
        (out / filename).write_bytes(data); return f"/static/notebook/{filename}"

args = gp.default_args(max_per_detector=2, workers=4)
payload = gp.build_payload(target, TempStorage(), args, cached=False,
                           progress=lambda stage, f, msg: print(f"{f:4.0%} {msg}"))
for e in payload["timelapse_epochs"]:
    print(e["epoch_id"], e["date"], e["n_exposures"], "exposures, filters:", list(e["chemical_filters"]))
for n in range(1, len(payload["timelapse_epochs"]) + 1):
    display(Image(out / f"{target.slug}_epoch{n}_rgb.png", width=250))
display(Image(out / f"{target.slug}_diff.png", width=250))

In [ ]:
diff = payload["difference_imaging"]
print(diff["detection_label"], "-", diff["detection_notice"])
for a in diff["detected_anomalies"]:
    print(a["anomaly_id"], a["kind"], f"SNR {a['snr_confidence']}", a["contamination_flags"], "clean" if a["is_clean_detection"] else "check")
spec_ = payload["spectrum_102_bands"]
print(len(spec_), "of 102 channels; saturated:", payload["saturated"], payload["spectrum_message"] or "")
print(payload["light_curve"])

### Notes
- Bright stars (Barnard's Star, R Draconis) saturate SPHEREx: their cores carry OVERFLOW/BLOOM. The pipeline shows
  them in the images but excludes them from spectra and light curves (`saturated: true`, values `null`), instead of inventing numbers.
- The anomaly detector is classic image processing (difference + threshold + contours), labelled BETA. Every candidate needs human verification.
- Data: SPHEREx DR1 (DOI 10.26131/IRSA678) and QR3 (DOI 10.26131/IRSA662) at IRSA/IPAC.